In [2]:
pip install torch_snippets

In [3]:
# bibliotecas
from torchvision.models import vgg19, VGG19_Weights
from torch_snippets import *
import torch.optim as optim

In [4]:
# pré treinada para extrair features
vgg = vgg19(weights=VGG19_Weights.DEFAULT).features.to(device).eval()
for param in vgg.parameters(): param.requires_grad = False

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:09<00:00, 58.7MB/s]


In [5]:
# funções de suporte
class GramMatrix(nn.Module):
    def forward(self, x):
        b, c, h, w = x.size()
        f = x.view(b, c, h*w)
        return torch.bmm(f, f.transpose(1, 2)) / (c * h * w)

In [6]:
# definição das camadas de interesse
content_layers = [21] # camadas profundas
style_layers = [0, 5, 10, 19, 28] # camadas variadas para estilo

In [16]:
# função principal
def stylize(content_img, style_img, iters=200):
    opt_img = content_img.clone().requires_grad_(True)
    optimizer = optim.Adam([opt_img], lr=0.02)
    mse = nn.MSELoss()
    gm = GramMatrix()

    # extrair alvos
    with torch.no_grad():
        target_content = [vgg[:l+1](content_img) for l in content_layers]
        target_style = [gm(vgg[:l+1](style_img)) for l in style_layers]

    for i in range(iters):
        optimizer.zero_grad()

        # extrair features
        out_content = [vgg[:l+1](opt_img) for l in content_layers]
        out_style = [gm(vgg[:l+1](opt_img)) for l in style_layers]

        # perdas
        loss_c = sum([mse(o, t) for o, t in zip(out_content, target_content)])
        loss_s = sum([mse(o, t) for o, t in zip(out_style, target_style)])

        total_loss = loss_c + (loss_s * 1e6)
        total_loss.backward()
        optimizer.step()

        if (i+1) % 50 == 0:
            print(f" {i+1} | Loss: {total_loss.item():.2f}")

    return opt_img.detach()